In [0]:
import mlflow
import mlflow.spark
import os
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler

# Récupère la session Spark active
spark = SparkSession.builder.getOrCreate()

# Récupère dbutils de façon compatible (Databricks SDK / PySpark)
try:
    from databricks.sdk.runtime import dbutils
except ImportError:
    try:
        from pyspark.dbutils import DBUtils
        dbutils = DBUtils(spark)
    except ImportError:
        pass  # En environnement Databricks natif, dbutils est déjà injecté

# Obligatoire en Free Edition pour SparkML + UC
os.environ["MLFLOW_DFS_TMP"] = "/Volumes/main/ml/models_volume/tmp"
dbutils.fs.mkdirs("/Volumes/main/ml/models_volume/tmp")

# Charger le modèle depuis Unity Catalog
model_uri = "models:/main.ml.fraud_detection_model/2"

model = mlflow.spark.load_model(
    model_uri,
    dfs_tmpdir="/Volumes/main/ml/models_volume/tmp"
)

print("Modèle MLflow chargé avec succès :", model_uri)

In [0]:
# Lecture du flux Silver Streaming
silver_stream = (
    spark.readStream
    .table("main.silver.transactions_silver_stream")
)

# Préparation des features avec VectorAssembler
feature_cols = [f"V{i}" for i in range(1, 29)] + ["amount"]

assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

stream_features = assembler.transform(silver_stream)

# Prédiction sur le flux Silver Streaming
pred_stream = model.transform(stream_features)

# Choix des colonnes pour la table de prédiction
predictions = (
    pred_stream
    .select(
        "time",
        "amount",
        "prediction",
        "probability",
        "is_fraud"
    )
)

predictions_stream_checkpoint = (
    "/Volumes/main/gold/gold_volume"
    "/_checkpoints/predictions_stream"
)

# Ecriture en streaming dans une table Delta Gold
predictions.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{predictions_stream_checkpoint}") \
    .trigger(once=True) \
    .table("main.gold.fraud_predictions_stream")